## ACID & Time Travel

Notebook 18 introduced the Delta Lake transaction log and showed how ACID guarantees and time travel work at a conceptual level. This notebook goes deep on the **mutation API** and the full time-travel feature set:

- **`DELETE FROM`** — remove rows matching a condition
- **`UPDATE SET`** — modify column values in place
- **`MERGE INTO`** — upsert: insert new rows, update existing ones, delete removed ones — all in one atomic operation
- **Time travel** — querying past versions and timestamps, SQL syntax
- **Rollback** — restoring a table to a previous version
- **`VACUUM`** — removing old file versions and controlling retention
- **Change Data Feed (CDF)** — streaming row-level changes out of a Delta table
- **Optimistic concurrency** — how Delta handles concurrent writers

```
Version 0  →  Version 1  →  Version 2  →  Version 3  →  current
  (CREATE)     (APPEND)      (UPDATE)      (MERGE)        ...
     │             │             │             │
     └─────────────┴─────────────┴─────────────┘
               Time travel: query any version
               Rollback:    restore any version as current
```

In [ ]:
import os, json, shutil, time
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, lit, current_timestamp,
    to_date, expr, when, concat
)
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("ACIDTimeTravelDeltaLake")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE    = os.path.dirname(os.path.abspath("19-acid-time-travel.ipynb"))
DATA    = os.path.join(BASE, "data")
SCRATCH = os.path.join(BASE, "_scratch_19")
os.makedirs(SCRATCH, exist_ok=True)

print(f"Spark {spark.version} ready")

In [ ]:
# Load source data
customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("merchant",    StringType(),      True),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.option("multiLine", "true").schema(StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])).json(f"{DATA}/loan_accounts")

print(f"customers  : {customers.count():>5} rows")
print(f"card_txns  : {card_txns.count():>5} rows")
print(f"loans      : {loan_accounts.count():>5} rows")

### Setup — Build a Versioned Delta Table

We will work with a `loans` Delta table throughout this notebook, building up a version history with each operation so time-travel queries have multiple past states to read.

In [ ]:
LOANS_PATH = os.path.join(SCRATCH, "loans")

# Version 0 — initial load
(
    loan_accounts
    .write
    .format("delta")
    .mode("overwrite")
    .save(LOANS_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS loans
    USING delta
    LOCATION '{LOANS_PATH}'
""")

loans_dt = DeltaTable.forPath(spark, LOANS_PATH)

v0_count = spark.read.format("delta").load(LOANS_PATH).count()
print(f"Version 0: {v0_count} rows")
spark.sql("SELECT * FROM loans LIMIT 5").show()

### DELETE FROM

`DELETE FROM` removes rows matching a predicate. Under the hood Delta does **not** physically delete rows from Parquet files — it rewrites the affected files with the matching rows omitted and records the old files as `remove` entries in the transaction log. The old files remain on disk until `VACUUM` cleans them up, which is what enables time travel.

```
DELETE FROM loans WHERE status = 'REJECTED'

Delta steps:
  1. Identify files that contain rows matching the predicate
  2. Rewrite those files without the matching rows
  3. Log: remove old files, add new files  ← one atomic transaction
  4. Old files kept on disk until VACUUM
```

In [ ]:
# Check current status distribution before delete
print("Before DELETE:")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()

# Version 1 — delete REJECTED loans (regulatory cleanup: rejected applications must be purged)
loans_dt.delete(condition=col("status") == "REJECTED")

print("After DELETE (version 1):")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()

In [ ]:
# SQL syntax — identical result
# spark.sql("DELETE FROM loans WHERE status = 'REJECTED'")

# Confirm via history
loans_dt.history().select("version", "timestamp", "operation", "operationParameters").show(5, truncate=False)

### UPDATE SET

`UPDATE` modifies column values for rows matching a predicate. Like DELETE, it rewrites the affected Parquet files — unaffected files are not touched.

You can update multiple columns in a single statement. The condition can reference any column, and the new value can be any expression.

```sql
UPDATE loans
SET    status = 'CLOSED',
       interest_rate = interest_rate * 0.9   -- 10% discount on close
WHERE  status = 'ACTIVE'
AND    disbursed_on < '2022-01-01'
```

In [ ]:
# Check ACTIVE loans before update
print("ACTIVE loans before UPDATE:")
spark.sql("""
    SELECT loan_id, customer_id, loan_type, status, interest_rate
    FROM   loans
    WHERE  status = 'ACTIVE'
    LIMIT  5
""").show()

# Version 2 — mark old disbursed loans as CLOSED and apply a rate reduction
loans_dt.update(
    condition = (col("status") == "ACTIVE") & (col("disbursed_on") < "2022-01-01"),
    set = {
        "status":        lit("CLOSED"),
        "interest_rate": (col("interest_rate") * 0.9),
    }
)

print("After UPDATE (version 2):")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()

print("Spot-check updated rows:")
spark.sql("""
    SELECT loan_id, status, interest_rate
    FROM   loans
    WHERE  status = 'CLOSED'
    LIMIT  5
""").show()

In [ ]:
# SQL syntax for UPDATE
spark.sql("""
    UPDATE loans
    SET    status = 'UNDER_REVIEW'
    WHERE  status = 'DELINQUENT'
    AND    interest_rate > 12
""")

# Version 3 created
print("After second UPDATE (version 3):")
spark.sql("SELECT status, count(*) AS n FROM loans GROUP BY status ORDER BY n DESC").show()

### MERGE INTO — Upsert

`MERGE INTO` is the most powerful Delta mutation. It joins a **source** DataFrame against the **target** table on a match condition and then applies different actions depending on whether a row matched:

```sql
MERGE INTO target t
USING  source s
ON     t.id = s.id                     ← match condition

WHEN MATCHED AND s.amount > 1000       ← matched + extra condition
  THEN UPDATE SET t.status = 'FLAGGED'

WHEN MATCHED                           ← matched (no extra condition)
  THEN UPDATE SET *                    ← update all columns from source

WHEN NOT MATCHED                       ← in source but not in target
  THEN INSERT *                        ← insert the new row

WHEN NOT MATCHED BY SOURCE             ← in target but not in source
  THEN DELETE                          ← delete the removed row
```

All clauses are optional. You can have zero or more `WHEN MATCHED` clauses (with different conditions), zero or one `WHEN NOT MATCHED`, and zero or one `WHEN NOT MATCHED BY SOURCE`.

**Common patterns:**
- **Upsert (SCD Type 1)** — update if exists, insert if new
- **Append if new** — insert only rows that don't already exist (deduplication)
- **Full sync** — update matching, insert new, delete removed (mirror a source table)
- **Conditional update** — update only if a business condition is met (e.g., newer timestamp)

In [ ]:
# Pattern 1 — SCD Type 1 upsert
# Incoming batch: some existing loans with updated fields, some new loans

from pyspark.sql.functions import round as _round

# Simulate an incoming delta: update 3 existing loans + add 2 brand new ones
existing_ids = (
    spark.sql("SELECT loan_id FROM loans WHERE status = 'ACTIVE' LIMIT 3")
    .collect()
)

updates = [
    {"loan_id": row["loan_id"], "customer_id": "CUST0001",
     "loan_type": "PERSONAL", "principal": 50000.0,
     "interest_rate": 11.5, "tenure_months": 36,
     "disbursed_on": "2024-06-01", "status": "ACTIVE"}
    for row in existing_ids
]
new_rows = [
    {"loan_id": "LN-NEW-001", "customer_id": "CUST0099",
     "loan_type": "HOME", "principal": 2500000.0,
     "interest_rate": 8.75, "tenure_months": 240,
     "disbursed_on": "2024-09-15", "status": "ACTIVE"},
    {"loan_id": "LN-NEW-002", "customer_id": "CUST0100",
     "loan_type": "VEHICLE", "principal": 750000.0,
     "interest_rate": 9.5, "tenure_months": 60,
     "disbursed_on": "2024-10-01", "status": "ACTIVE"},
]

incoming_schema = StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DoubleType(),      False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  StringType(),      False),
    StructField("status",        StringType(),      False),
])

source_df = spark.createDataFrame(updates + new_rows, schema=incoming_schema)

count_before = spark.sql("SELECT count(*) FROM loans").collect()[0][0]

# MERGE: update existing loans, insert new ones  (version 4)
(
    loans_dt.alias("t")
    .merge(
        source_df.alias("s"),
        "t.loan_id = s.loan_id"
    )
    .whenMatchedUpdateAll()     # update every column from source when loan_id matches
    .whenNotMatchedInsertAll()  # insert new rows when loan_id not found in target
    .execute()
)

count_after = spark.sql("SELECT count(*) FROM loans").collect()[0][0]
print(f"Rows before MERGE : {count_before}")
print(f"Rows after MERGE  : {count_after}  (+{count_after - count_before} new rows)")

# Confirm new rows exist
spark.sql("""
    SELECT loan_id, customer_id, loan_type, principal, status
    FROM   loans
    WHERE  loan_id IN ('LN-NEW-001', 'LN-NEW-002')
""").show()

In [ ]:
# Pattern 2 — conditional update: only update if source has a higher interest rate
# (prevents a stale source from overwriting a more recent update)

stale_source = spark.createDataFrame([
    {"loan_id": "LN-NEW-001", "interest_rate": 7.0, "status": "ACTIVE"},  # lower rate — should NOT overwrite
    {"loan_id": "LN-NEW-002", "interest_rate": 10.5, "status": "ACTIVE"}, # higher rate — SHOULD overwrite
], schema=StructType([
    StructField("loan_id",       StringType(), False),
    StructField("interest_rate", DoubleType(), False),
    StructField("status",        StringType(), False),
]))

# Version 5 — conditional update
(
    loans_dt.alias("t")
    .merge(stale_source.alias("s"), "t.loan_id = s.loan_id")
    .whenMatchedUpdate(
        condition = col("s.interest_rate") > col("t.interest_rate"),  # only if source rate is higher
        set = {"interest_rate": col("s.interest_rate")}
    )
    .execute()
)

print("After conditional MERGE:")
spark.sql("""
    SELECT loan_id, interest_rate
    FROM   loans
    WHERE  loan_id IN ('LN-NEW-001', 'LN-NEW-002')
""").show()
print("LN-NEW-001 kept 8.75 (source 7.0 was lower — not applied)")
print("LN-NEW-002 updated to 10.5 (source 10.5 > current 9.5)")

In [ ]:
# Pattern 3 — full sync: update matching, insert new, DELETE rows not in source
# Mirrors a source system: rows removed from the source are deleted from the target

SYNC_PATH = os.path.join(SCRATCH, "loans_sync")
loan_accounts.limit(20).write.format("delta").mode("overwrite").save(SYNC_PATH)
sync_dt = DeltaTable.forPath(spark, SYNC_PATH)

before = spark.read.format("delta").load(SYNC_PATH).count()
print(f"Sync table before: {before} rows")

# Source: only 15 of the original 20 loans (5 were "removed" from the source system)
source_15 = loan_accounts.limit(15)

(
    sync_dt.alias("t")
    .merge(source_15.alias("s"), "t.loan_id = s.loan_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()  # delete rows in target not present in source
    .execute()
)

after = spark.read.format("delta").load(SYNC_PATH).count()
print(f"Sync table after:  {after} rows (5 deleted by NOT MATCHED BY SOURCE)")

### MERGE INTO — SQL Syntax

The same operations are available directly in Spark SQL. This is useful when building pipelines with `spark.sql()` or when writing transformations in a SQL-first style.

In [ ]:
# Create a staging table with incoming customer updates
CUST_DELTA = os.path.join(SCRATCH, "customers")
customers.write.format("delta").mode("overwrite").save(CUST_DELTA)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS customers_delta
    USING delta LOCATION '{CUST_DELTA}'
""")

# Staging: some customers changed city/email, one is brand new
staging = spark.createDataFrame([
    ("CUST0001", "Alice Johnson", "alice.new@example.com", "555-0001",
     "New York", "NY", None, True, datetime(2024, 1, 1)),
    ("CUST0002", "Bob Smith",    "bob@example.com",      "555-0002",
     "Chicago",  "IL", None, False, datetime(2024, 1, 1)),
    ("CUST9999", "New Customer", "new@example.com",      "555-9999",
     "Austin",   "TX", None, True,  datetime(2024, 10, 1)),  # new
], schema=StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
]))

staging.createOrReplaceTempView("customer_staging")

spark.sql("""
    MERGE INTO customers_delta AS t
    USING  customer_staging   AS s
    ON     t.customer_id = s.customer_id

    WHEN MATCHED THEN
      UPDATE SET
        t.full_name    = s.full_name,
        t.email        = s.email,
        t.city         = s.city,
        t.kyc_verified = s.kyc_verified

    WHEN NOT MATCHED THEN
      INSERT (customer_id, full_name, email, phone,
              city, state, date_of_birth, kyc_verified, created_at)
      VALUES (s.customer_id, s.full_name, s.email, s.phone,
              s.city, s.state, s.date_of_birth, s.kyc_verified, s.created_at)
""")

print("After SQL MERGE:")
spark.sql("""
    SELECT customer_id, full_name, email, city, kyc_verified
    FROM   customers_delta
    WHERE  customer_id IN ('CUST0001', 'CUST0002', 'CUST9999')
""").show(truncate=False)

### Time Travel — Full API

Delta retains old data files on disk until `VACUUM` removes them. This means you can query any past version of the table — not just the current one.

**Two ways to specify the past state:**

| Method | DataFrame API | SQL |
|---|---|---|
| By version number | `.option("versionAsOf", N)` | `VERSION AS OF N` |
| By timestamp | `.option("timestampAsOf", "YYYY-MM-DD HH:MM:SS")` | `TIMESTAMP AS OF 'YYYY-MM-DD'` |

The version number comes from the transaction log (0-indexed, one per commit). The timestamp must fall within the retention window (default 30 days, controlled by `delta.logRetentionDuration`).

In [ ]:
# Show full version history of the loans table
print("=== LOANS TABLE HISTORY ===")
loans_dt.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

In [ ]:
# Time travel by version — DataFrame API
v0 = spark.read.format("delta").option("versionAsOf", 0).load(LOANS_PATH)
v1 = spark.read.format("delta").option("versionAsOf", 1).load(LOANS_PATH)  # after DELETE
v2 = spark.read.format("delta").option("versionAsOf", 2).load(LOANS_PATH)  # after first UPDATE
v_current = spark.read.format("delta").load(LOANS_PATH)                     # current

print(f"Version 0  (initial load)        : {v0.count()} rows")
print(f"Version 1  (after DELETE)        : {v1.count()} rows")
print(f"Version 2  (after UPDATE CLOSED) : {v2.count()} rows")
print(f"Current    (after all ops)       : {v_current.count()} rows")

# Confirm REJECTED loans visible in v0 but gone from v1
print(f"\nREJECTED loans in v0 : {v0.filter(col('status') == 'REJECTED').count()}")
print(f"REJECTED loans in v1 : {v1.filter(col('status') == 'REJECTED').count()} ← deleted")

In [ ]:
# Time travel by version — SQL syntax
spark.sql("SELECT status, count(*) AS n FROM loans VERSION AS OF 0 GROUP BY status ORDER BY n DESC").show()
spark.sql("SELECT status, count(*) AS n FROM loans VERSION AS OF 2 GROUP BY status ORDER BY n DESC").show()

In [ ]:
# Time travel by timestamp — find the commit timestamps from the log
history_rows = loans_dt.history().select("version", "timestamp").orderBy("version").collect()

print("Commit timestamps:")
for row in history_rows:
    print(f"  v{row['version']}  {row['timestamp']}")

# Query as-of the timestamp of version 1 (after DELETE)
if len(history_rows) > 1:
    ts_v1 = history_rows[1]["timestamp"]
    ts_str = ts_v1.strftime("%Y-%m-%d %H:%M:%S") if hasattr(ts_v1, 'strftime') else str(ts_v1)
    print(f"\nQuery as of v1 timestamp: {ts_str}")
    spark.sql(f"""
        SELECT status, count(*) AS n
        FROM   loans TIMESTAMP AS OF '{ts_str}'
        GROUP  BY status
        ORDER  BY n DESC
    """).show()

### Rollback — Restoring a Previous Version

Delta does not have a single-command rollback, but the pattern is simple: **time-travel read the past version and overwrite the current table** with it. This creates a new transaction (the highest version number) that brings the table back to the past state.

The rollback is itself recorded in the transaction log, so the history of both the mistake and the recovery is preserved.

```
v0 → v1 → v2 (bad write) → v3 (rollback to v0)
                              ↑
              new transaction; table content = v0 content
              but v1 and v2 are still in the history
```

**Alternative — `RESTORE` command (Databricks / Delta 1.2+):**
```sql
RESTORE TABLE loans TO VERSION AS OF 0
RESTORE TABLE loans TO TIMESTAMP AS OF '2024-01-01 09:00:00'
```

In [ ]:
ROLLBACK_PATH = os.path.join(SCRATCH, "loans_rollback")

# Version 0: initial data
loan_accounts.write.format("delta").mode("overwrite").save(ROLLBACK_PATH)
rb_dt = DeltaTable.forPath(spark, ROLLBACK_PATH)
print(f"v0 count: {spark.read.format('delta').load(ROLLBACK_PATH).count()}")

# Version 1: bad delete — someone accidentally removed all PERSONAL loans
rb_dt.delete(condition=col("loan_type") == "PERSONAL")
print(f"v1 count (bad delete): {spark.read.format('delta').load(ROLLBACK_PATH).count()}")
personal_after = spark.read.format("delta").load(ROLLBACK_PATH).filter(col("loan_type") == "PERSONAL").count()
print(f"  PERSONAL loans remaining: {personal_after}  ← should have been kept!")

# Rollback: restore to version 0
v0_snapshot = spark.read.format("delta").option("versionAsOf", 0).load(ROLLBACK_PATH)
v0_snapshot.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ROLLBACK_PATH)

# Version 2 is now the rollback — same content as v0
print(f"\nv2 count (after rollback): {spark.read.format('delta').load(ROLLBACK_PATH).count()}")
personal_restored = spark.read.format("delta").load(ROLLBACK_PATH).filter(col("loan_type") == "PERSONAL").count()
print(f"  PERSONAL loans restored : {personal_restored}")

# Full history shows the mistake AND the recovery
DeltaTable.forPath(spark, ROLLBACK_PATH).history() \
    .select("version", "timestamp", "operation").show()

In [ ]:
# RESTORE TABLE command (Delta 1.2+ / Databricks)
# Equivalent single-command rollback — creates a new version like the overwrite pattern above

RESTORE_PATH = os.path.join(SCRATCH, "loans_restore")
loan_accounts.write.format("delta").mode("overwrite").save(RESTORE_PATH)
DeltaTable.forPath(spark, RESTORE_PATH).delete(col("loan_type") == "PERSONAL")  # bad delete

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS loans_restore
    USING delta LOCATION '{RESTORE_PATH}'
""")

before_restore = spark.sql("SELECT count(*) FROM loans_restore").collect()[0][0]
print(f"Before RESTORE: {before_restore} rows")

spark.sql("RESTORE TABLE loans_restore TO VERSION AS OF 0")

after_restore = spark.sql("SELECT count(*) FROM loans_restore").collect()[0][0]
print(f"After RESTORE : {after_restore} rows")

spark.sql("DESCRIBE HISTORY loans_restore") \
    .select("version", "timestamp", "operation").show()

### VACUUM — Removing Old Files

When Delta writes a new version of data it marks old files as `remove` in the log but **does not delete them from disk**. This is intentional — those files are what enable time travel.

`VACUUM` physically deletes files that are:
1. Marked as removed in the log, **and**
2. Older than the **retention threshold** (default: 7 days)

```
VACUUM loans RETAIN 168 HOURS   -- keep 7 days (default)
VACUUM loans RETAIN 0 HOURS     -- delete everything beyond current version (dangerous!)
```

**After VACUUM:**
- Disk space is freed
- Time travel to versions whose files were deleted becomes impossible
- The log entries still exist (history is preserved) — but the actual data files are gone

**Safety check:** Delta blocks `VACUUM` with a retention period below 7 days by default. Override with `spark.databricks.delta.retentionDurationCheck.enabled = false` (for testing only — never in production).

In [ ]:
# Count files before VACUUM
def count_parquet_files(path: str) -> int:
    total = 0
    for root, dirs, files in os.walk(path):
        if "_delta_log" in root:
            continue
        total += sum(1 for f in files if f.endswith(".parquet"))
    return total

print(f"Parquet files before VACUUM: {count_parquet_files(LOANS_PATH)}")

# DRY RUN — list files VACUUM would delete without actually deleting them
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
dry_run = spark.sql(f"VACUUM loans RETAIN 0 HOURS DRY RUN")
print(f"Files VACUUM would remove: {dry_run.count()}")
dry_run.show(5, truncate=False)

In [ ]:
# Execute VACUUM with 0-hour retention (testing only — removes all old versions)
spark.sql("VACUUM loans RETAIN 0 HOURS")

print(f"Parquet files after VACUUM: {count_parquet_files(LOANS_PATH)}")

# Time travel to an old version now fails — files are gone
try:
    old = spark.read.format("delta").option("versionAsOf", 0).load(LOANS_PATH).count()
    print(f"Version 0 rows: {old}")
except Exception as e:
    print("Time travel to v0 fails (files removed by VACUUM):")
    print(f"  {str(e)[:160]}")

# But current version is still fully readable
print(f"Current version rows: {spark.read.format('delta').load(LOANS_PATH).count()}")

# Restore the safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

### Table Properties — Controlling Retention

Two table properties control how long history and data files are retained:

| Property | Default | Controls |
|---|---|---|
| `delta.logRetentionDuration` | `interval 30 days` | How long transaction log entries are kept |
| `delta.deletedFileRetentionDuration` | `interval 7 days` | How long removed data files are kept on disk |

Set them per table with `ALTER TABLE SET TBLPROPERTIES` or at write time with `.option()`.

```sql
ALTER TABLE loans SET TBLPROPERTIES (
  'delta.logRetentionDuration'        = 'interval 90 days',
  'delta.deletedFileRetentionDuration' = 'interval 30 days'
);
```

Longer retention = more time travel range + more storage cost.
Shorter retention = less storage + smaller time travel window.

In [ ]:
# Set retention properties on the loans table
spark.sql("""
    ALTER TABLE loans SET TBLPROPERTIES (
        'delta.logRetentionDuration'         = 'interval 90 days',
        'delta.deletedFileRetentionDuration'  = 'interval 30 days'
    )
""")

# Verify the properties are set
spark.sql("SHOW TBLPROPERTIES loans").filter(
    col("key").startswith("delta")
).show(truncate=False)

### Change Data Feed (CDF)

**Change Data Feed** tracks row-level changes to a Delta table — which rows were inserted, updated (before and after images), or deleted. CDF is the foundation for CDC (Change Data Capture) pipelines: instead of re-reading the entire table every batch, downstream consumers read only what changed.

**Enable CDF** per table:
```sql
ALTER TABLE loans SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
```

**Read changes** between two versions:
```python
spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 2) \
    .option("endingVersion",   4) \
    .load(path)
```

CDF adds three system columns to each row:

| Column | Values | Meaning |
|---|---|---|
| `_change_type` | `insert` / `update_preimage` / `update_postimage` / `delete` | What happened to this row |
| `_commit_version` | integer | Which table version this change belongs to |
| `_commit_timestamp` | timestamp | When the commit happened |

In [ ]:
CDF_PATH = os.path.join(SCRATCH, "loans_cdf")

# Create the table with CDF enabled from the start
(
    loan_accounts
    .write
    .format("delta")
    .mode("overwrite")
    .option("delta.enableChangeDataFeed", "true")
    .save(CDF_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS loans_cdf
    USING delta LOCATION '{CDF_PATH}'
""")

cdf_dt = DeltaTable.forPath(spark, CDF_PATH)

# Make a series of changes — each creates a new version
# v1: delete REJECTED
cdf_dt.delete(col("status") == "REJECTED")

# v2: update ACTIVE → flag ones with high interest
spark.sql("""
    UPDATE loans_cdf
    SET    status = 'HIGH_RISK'
    WHERE  status = 'ACTIVE' AND interest_rate > 14
""")

# v3: insert two new loans
new_loans = spark.createDataFrame([
    ("LN-CDF-001", "CUST0001", "PERSONAL", 100000.0, 10.5, 24, "2024-11-01", "ACTIVE"),
    ("LN-CDF-002", "CUST0002", "VEHICLE",   50000.0,  9.8, 48, "2024-11-01", "ACTIVE"),
], schema=StructType([
    StructField("loan_id",       StringType(), False),
    StructField("customer_id",   StringType(), False),
    StructField("loan_type",     StringType(), False),
    StructField("principal",     DoubleType(), False),
    StructField("interest_rate", DoubleType(), False),
    StructField("tenure_months", IntegerType(), False),
    StructField("disbursed_on",  StringType(), False),
    StructField("status",        StringType(), False),
]))
new_loans.write.format("delta").mode("append").save(CDF_PATH)

print("Changes made. Versions 0–3 created.")

In [ ]:
# Read ALL changes since version 1 (skip the initial load at v0)
changes = (
    spark.read
    .format("delta")
    .option("readChangeFeed",  "true")
    .option("startingVersion", 1)
    .load(CDF_PATH)
)

print("All changes (v1 onwards):")
changes.select(
    "_change_type", "_commit_version",
    "loan_id", "loan_type", "status", "interest_rate"
).orderBy("_commit_version", "_change_type").show(20, truncate=False)

# Summary of change types
print("Change type summary:")
changes.groupBy("_change_type", "_commit_version").count() \
    .orderBy("_commit_version", "_change_type").show()

In [ ]:
# CDF as a streaming source — downstream pipeline reads only new changes
cdf_stream = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed",  "true")
    .option("startingVersion", 1)
    .load(CDF_PATH)
)

cdf_q = (
    cdf_stream
    .select("_change_type", "_commit_version", "loan_id", "status", "interest_rate")
    .writeStream
    .format("memory").queryName("cdf_changes")
    .outputMode("append")
    .trigger(once=True)
    .start()
)
cdf_q.awaitTermination()

print("Streaming CDF result:")
spark.sql("""
    SELECT _change_type, _commit_version, count(*) AS n
    FROM   cdf_changes
    GROUP  BY _change_type, _commit_version
    ORDER  BY _commit_version, _change_type
""").show()

### Optimistic Concurrency — How Delta Handles Concurrent Writers

Delta Lake uses **optimistic concurrency control (OCC)**: multiple writers can run simultaneously without blocking each other. When a writer commits, Delta checks whether any conflicting change was committed since the writer started. If yes, the commit fails and the writer can retry.

**Conflict detection rules:**

| Operation | Conflicts with |
|---|---|
| Append | Rarely conflicts — appends are almost always compatible |
| Update / Delete | Conflicts if another writer touched the same files |
| MERGE | Conflicts if another writer modified files that MERGE needs to read or write |
| Schema change | Conflicts with any concurrent writer |

**Partition-level isolation**: if two writers update different partitions, Delta can detect this and allow both to commit without conflict.

```
Writer A: UPDATE loans WHERE loan_type = 'HOME'    → touches HOME partition files
Writer B: UPDATE loans WHERE loan_type = 'VEHICLE' → touches VEHICLE partition files

No file overlap → both commits succeed.

Writer C: UPDATE loans WHERE status = 'ACTIVE'     → touches ALL partitions
Writer D: UPDATE loans WHERE loan_type = 'HOME'    → starts while C is running

C commits first → D's commit fails (conflict on HOME partition files)
D retries with the latest version → succeeds
```

In [ ]:
# Inspect the protocol version — controls which reader/writer features are allowed
detail = DeltaTable.forPath(spark, LOANS_PATH).detail().collect()[0]
print("Table detail:")
print(f"  format           : {detail['format']}")
print(f"  numFiles         : {detail['numFiles']}")
print(f"  sizeInBytes      : {detail['sizeInBytes']:,}")

# Protocol version from the log
log_dir = os.path.join(LOANS_PATH, "_delta_log")
log_files = sorted(f for f in os.listdir(log_dir) if f.endswith(".json"))
for lf in log_files[:2]:
    with open(os.path.join(log_dir, lf)) as fh:
        for line in fh:
            entry = json.loads(line)
            if "protocol" in entry:
                print(f"\nProtocol in {lf}:")
                print(f"  minReaderVersion : {entry['protocol']['minReaderVersion']}")
                print(f"  minWriterVersion : {entry['protocol']['minWriterVersion']}")
                break

### Comparing Two Versions

A common operational task is finding exactly what changed between two table versions — which rows were added, which were modified, which were removed. Time travel makes this straightforward: read both versions as static DataFrames and compare them.

In [ ]:
# Rebuild the loans table with a clean history for this diff
DIFF_PATH = os.path.join(SCRATCH, "loans_diff")
loan_accounts.write.format("delta").mode("overwrite").save(DIFF_PATH)
diff_dt = DeltaTable.forPath(spark, DIFF_PATH)

# v1: update some interest rates
diff_dt.update(
    condition = col("loan_type") == "HOME",
    set = {"interest_rate": lit(8.5)}
)

# Compare v0 vs v1: find rows that changed
v0 = spark.read.format("delta").option("versionAsOf", 0).load(DIFF_PATH)
v1 = spark.read.format("delta").option("versionAsOf", 1).load(DIFF_PATH)

# Rows in v1 that differ from v0 on interest_rate
changed = (
    v1.alias("new")
    .join(v0.alias("old"), on="loan_id")
    .filter(col("new.interest_rate") != col("old.interest_rate"))
    .select(
        col("new.loan_id"),
        col("new.loan_type"),
        col("old.interest_rate").alias("old_rate"),
        col("new.interest_rate").alias("new_rate"),
    )
)

print(f"Rows changed between v0 and v1: {changed.count()}")
changed.show(10)

### Summary

**DELETE FROM** removes rows matching a predicate. Delta rewrites affected files and records `remove` entries in the log — old files stay on disk until `VACUUM`.

**UPDATE SET** modifies column values in place. Only files containing matching rows are rewritten. Multiple columns can be updated in one statement.

**MERGE INTO** is the upsert workhorse:
- `WHEN MATCHED THEN UPDATE` — update existing rows (optionally with a condition)
- `WHEN NOT MATCHED THEN INSERT` — insert rows new to the target
- `WHEN NOT MATCHED BY SOURCE THEN DELETE` — delete rows removed from the source
- All actions execute in a single atomic transaction

**Time travel** reads any past table state:
- `versionAsOf` / `VERSION AS OF N` — by version number
- `timestampAsOf` / `TIMESTAMP AS OF 'date'` — by timestamp
- Only works while old files are still on disk (before `VACUUM` removes them)

**Rollback** — read a past version and overwrite the current table (`RESTORE TABLE` for a single command on Databricks / Delta 1.2+).

**VACUUM** physically deletes old data files beyond the retention threshold (default 7 days). After VACUUM, time travel to removed versions is impossible but the history log entries remain.

**Table properties** — `delta.logRetentionDuration` and `delta.deletedFileRetentionDuration` control how long history and data files are kept.

**Change Data Feed** (`readChangeFeed=true`) exposes row-level changes (`insert`, `update_preimage`, `update_postimage`, `delete`) per commit. Works as a batch read or a streaming source — the foundation for CDC pipelines.

**Optimistic concurrency** — writers don't block each other. Conflicts are detected at commit time. Partition-level isolation allows non-overlapping writes to succeed concurrently.

In [ ]:
# Cleanup
for tbl in ["loans", "loans_cdf", "loans_restore", "customers_delta"]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

if os.path.exists(SCRATCH):
    shutil.rmtree(SCRATCH)
    print(f"Removed: {SCRATCH}")
print("Cleanup complete.")